# 05 — Simulação de Ingestão Streaming

Este notebook demonstra a ingestão em **tempo quase-real** de novos dados do indicador.

**Cenário simulado**: Ao longo do ano letivo de 2024, novas medições do
Indicador Criança Alfabetizada são publicadas pelo INEP para municípios,
à medida que as escolas concluem as avaliações do 2º ano.

**Arquitetura de streaming**:
```
[Eventos Simulados]
      │  (append em Delta table com CDF habilitado)
      ▼
origens.tc02_streaming_indicador_eventos  ← Producer (este notebook)
      │
      │  readStream (Delta Change Data Feed)
      ▼
bronze.tc02_streaming_indicador_raw       ← Consumer (Structured Streaming)
```

> **Nota**: Requer cluster Databricks com suporte a Structured Streaming.
> Não compatível com compute Serverless (use cluster Standard ou compute Jobs).

> Execute as células em sequência: primeiro o **Setup**, depois o **Producer**,
> depois o **Consumer**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json

## Setup: Tabelas de Streaming com Delta Change Data Feed

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS origens")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# Habilita Change Data Feed na tabela de origem para suportar readStream incremental
spark.sql("""
    CREATE TABLE IF NOT EXISTS origens.tc02_streaming_indicador_eventos (
        evento_id              STRING,
        tipo_evento            STRING,
        timestamp_evento       STRING,
        id_municipio           INT,
        nome_municipio         STRING,
        sigla_uf               STRING,
        ano                    INT,
        total_alunos_2o_ano    INT,
        alunos_alfabetizados   INT,
        indicador              DOUBLE,
        _sistema_origem        STRING,
        _data_criacao          TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# Tabela Bronze de destino do streaming
spark.sql("""
    CREATE TABLE IF NOT EXISTS bronze.tc02_streaming_indicador_raw (
        evento_id              STRING,
        tipo_evento            STRING,
        timestamp_evento       STRING,
        id_municipio           INT,
        nome_municipio         STRING,
        sigla_uf               STRING,
        ano                    INT,
        total_alunos_2o_ano    INT,
        alunos_alfabetizados   INT,
        indicador              DOUBLE,
        _sistema_origem        STRING,
        _data_criacao          TIMESTAMP,
        _data_ingestao_stream  TIMESTAMP,
        _operacao_cdf          STRING
    )
    USING DELTA
""")

print("Tabelas de streaming criadas/verificadas.")
print("  - origens.tc02_streaming_indicador_eventos (Delta CDF habilitado)")
print("  - bronze.tc02_streaming_indicador_raw")

## Producer: Lote 1 — Novas Medições de 2024

Simula o recebimento das primeiras medições do indicador referentes ao ano letivo 2024.

In [0]:
schema_evento = T.StructType([
    T.StructField("evento_id",           T.StringType(),  True),
    T.StructField("tipo_evento",         T.StringType(),  True),
    T.StructField("timestamp_evento",    T.StringType(),  True),
    T.StructField("id_municipio",        T.IntegerType(), True),
    T.StructField("nome_municipio",      T.StringType(),  True),
    T.StructField("sigla_uf",            T.StringType(),  True),
    T.StructField("ano",                 T.IntegerType(), True),
    T.StructField("total_alunos_2o_ano", T.IntegerType(), True),
    T.StructField("alunos_alfabetizados",T.IntegerType(), True),
    T.StructField("indicador",           T.DoubleType(),  True),
])

lote_1 = [
    ("stream-evt-001", "NOVO_INDICADOR_2024", "2024-03-15T10:30:00", 3550308, "São Paulo",      "SP", 2024, 186500, 145170, 77.84),
    ("stream-evt-002", "NOVO_INDICADOR_2024", "2024-03-15T10:31:00", 4314902, "Porto Alegre",   "RS", 2024,  18350,  15040, 81.96),
    ("stream-evt-003", "NOVO_INDICADOR_2024", "2024-03-15T10:32:00", 2111300, "São Luís",       "MA", 2024,  14200,   8949, 63.02),
    ("stream-evt-004", "NOVO_INDICADOR_2024", "2024-03-15T10:33:00", 1501402, "Belém",          "PA", 2024,  18700,  10472, 56.00),
    ("stream-evt-005", "NOVO_INDICADOR_2024", "2024-03-15T10:34:00", 3106200, "Belo Horizonte", "MG", 2024,  31200,  22464, 72.00),
]

df_lote_1 = (
    spark.createDataFrame(lote_1, schema=schema_evento)
    .withColumn("_sistema_origem", F.lit("streaming_simulado_lote_1"))
    .withColumn("_data_criacao", F.current_timestamp())
)

df_lote_1.write.format("delta").mode("append").saveAsTable("origens.tc02_streaming_indicador_eventos")

total_eventos = spark.read.table("origens.tc02_streaming_indicador_eventos").count()
print(f"Lote 1 publicado: {len(lote_1)} eventos")
print(f"Total na tabela de origem: {total_eventos} eventos")
display(spark.read.table("origens.tc02_streaming_indicador_eventos"))

## Consumer: Leitura com Structured Streaming (Delta CDF)

`trigger(availableNow=True)` processa todos os eventos disponíveis e para —
comportamento ideal para pipelines híbridos e demos.
Para streaming contínuo em produção, use `trigger(processingTime="30 seconds")`.

In [0]:
df_stream = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .table("origens.tc02_streaming_indicador_eventos")
)

df_stream_bronze = (
    df_stream
    .filter(F.col("_change_type").isin("insert", "update_postimage"))
    .withColumn("_data_ingestao_stream", F.current_timestamp())
    .withColumn("_operacao_cdf", F.col("_change_type"))
    .drop("_change_type", "_commit_version", "_commit_timestamp")
)

query = (
    df_stream_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "dbfs:/checkpoints/tc02_streaming_bronze")
    .trigger(availableNow=True)
    .toTable("bronze.tc02_streaming_indicador_raw")
)

query.awaitTermination()

total_bronze_stream = spark.read.table("bronze.tc02_streaming_indicador_raw").count()
print(f"Streaming concluído (trigger.availableNow).")
print(f"Bronze Streaming: {total_bronze_stream} registros ingeridos")
display(spark.read.table("bronze.tc02_streaming_indicador_raw"))

## Producer: Lote 2 — Revisão e Novas Medições

Simula chegada de novos eventos após o consumer já ter processado o Lote 1.
Inclui uma **revisão** de dado (tipo `REVISAO_INDICADOR`) e novos municípios.

In [0]:
lote_2 = [
    ("stream-evt-006", "NOVO_INDICADOR_2024",  "2024-04-01T09:00:00", 4209102, "Florianópolis",  "SC", 2024,   6800,   5746, 84.50),
    ("stream-evt-007", "NOVO_INDICADOR_2024",  "2024-04-01T09:01:00", 2304400, "Fortaleza",      "CE", 2024,  32500,  20475, 63.00),
    ("stream-evt-008", "NOVO_INDICADOR_2024",  "2024-04-01T09:02:00", 2611606, "Recife",         "PE", 2024,  20400,  12444, 61.00),
    ("stream-evt-009", "REVISAO_INDICADOR",    "2024-04-01T09:10:00", 3550308, "São Paulo",      "SP", 2024, 186500, 145879, 78.22),
    ("stream-evt-010", "NOVO_INDICADOR_2024",  "2024-04-01T09:15:00", 1302603, "Manaus",         "AM", 2024,  27800,  15929, 57.30),
]

df_lote_2 = (
    spark.createDataFrame(lote_2, schema=schema_evento)
    .withColumn("_sistema_origem", F.lit("streaming_simulado_lote_2"))
    .withColumn("_data_criacao", F.current_timestamp())
)

df_lote_2.write.format("delta").mode("append").saveAsTable("origens.tc02_streaming_indicador_eventos")

print(f"Lote 2 publicado: {len(lote_2)} eventos (inclui 1 revisão)")
print(f"Total na tabela de origem: {spark.read.table('origens.tc02_streaming_indicador_eventos').count()} eventos")

## Consumer: Re-execução para Processar Lote 2

O checkpoint garante que apenas os **novos eventos** (Lote 2) sejam processados.

In [0]:
query2 = (
    df_stream_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "dbfs:/checkpoints/tc02_streaming_bronze")
    .trigger(availableNow=True)
    .toTable("bronze.tc02_streaming_indicador_raw")
)

query2.awaitTermination()

total_final = spark.read.table("bronze.tc02_streaming_indicador_raw").count()
print(f"Streaming Lote 2 concluído.")
print(f"Bronze Streaming total: {total_final} registros")
display(
    spark.read.table("bronze.tc02_streaming_indicador_raw")
    .orderBy("_data_ingestao_stream")
)

## Verificação: Dados de Streaming na Bronze

Os dados de streaming e batch coexistem na Bronze. A Silver pode integrar ambos.

In [0]:
print("=" * 50)
print("RESUMO — INGESTÃO STREAMING")
print("=" * 50)
print(f"\nEventos produzidos (origem): {spark.read.table('origens.tc02_streaming_indicador_eventos').count()}")
print(f"Eventos consumidos (bronze): {spark.read.table('bronze.tc02_streaming_indicador_raw').count()}")

por_tipo = (
    spark.read.table("bronze.tc02_streaming_indicador_raw")
    .groupBy("tipo_evento")
    .count()
    .orderBy("tipo_evento")
)
display(por_tipo)

por_uf = (
    spark.read.table("bronze.tc02_streaming_indicador_raw")
    .groupBy("sigla_uf", "ano")
    .agg(F.count("*").alias("eventos"), F.avg("indicador").alias("indicador_medio"))
    .orderBy("sigla_uf")
)
display(por_uf)

print("\nPipeline Streaming demonstrado com sucesso!")
print("Os dados de streaming já estão na Bronze e podem ser integrados na Silver/Gold.")